In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# Automatically determine the project root (the folder containing "main")
root = Path.cwd().parent    # current dir = main/second → parent = main
sys.path.append(str(root))

In [8]:
import os
os.environ['DATABASE_URL'] = 'sqlite:///resumes.db'
import torch
import pandas as pd
from datasets import load_from_disk
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch import nn
import numpy as np
from transformers import RobertaTokenizer, RobertaModel, BatchEncoding

from embeddings.contrastive_learning import AUGMENTATION_FNS, ContrastiveLearningModel, ContrastiveLearningDataset, load_model
from infrastructure.database import init_db, import_postings_from_csv, import_resumes_from_csv
from embeddings.embed_stage2 import (
    fetch_all_postings_text, 
    fetch_all_resumes_text, 
    embed_text, 
    doc_sim_score
)

In [9]:
TOKENIZER = RobertaTokenizer.from_pretrained('roberta-base')
DEVICE = ('cuda:0' if torch.cuda.is_available() else 'cpu')

In [10]:
posting_db_url = 'sqlite:///postings.db'
resume_db_url = 'sqlite:///resumes.db'

In [11]:
# fetch_all_resumes_text(resume_db_url)

In [12]:
# init_db
# import_postings_from_csv()

In [13]:
# init_db()
# import_resumes_from_csv(csv_path='resume_data/sample_resumes.csv')

In [14]:
'''
How to get a single document similarity score from a list of word embeddings from both job and resume?

- Unweighted average
- K-means cluster --> concatenate cluster centers
- Soft k-means cluster
- Discrete cosine transform

'''

'\nHow to get a single document similarity score from a list of word embeddings from both job and resume?\n\n- Unweighted average\n- K-means cluster --> concatenate cluster centers\n- Soft k-means cluster\n- Discrete cosine transform\n\n'

In [15]:
postings = fetch_all_postings_text(posting_db_url)
# resumes = fetch_all_resumes_text(resume_db_url)
df_resumes = pd.read_csv('resume_data/Resume.csv')

In [16]:
resumes = df_resumes['Resume_str'].tolist()

In [17]:
# postings[4]

In [18]:
# resumes[4]

In [22]:
# (11,4), (2,4)
# bert_model = RobertaModel.from_pretrained('./roberta-tuned-v1', add_pooling_layer=False, output_hidden_states=True)
# bert_model = RobertaModel.from_pretrained('roberta-base', add_pooling_layer=False, output_hidden_states=True)
# cl_model = ContrastiveLearningModel(bert_model, out_embed_dim=588).to('cuda:0')
# cl_model = load_model(cl_model, './temp/contrastive_learning.pth')

scores = []

for i in (list(range(len(postings)))):
#     if i != 48:
#         continue
    for j in tqdm(list(range(len(resumes)))):
        score = doc_sim_score(postings[3], resumes[j], device=DEVICE, dist_func='soft_align')
        scores.append((i, j, score))
#         break
    break

#     break
# p = embed_text(resumes[8], 'concat_last_four')
# r = embed_text(resumes[9], 'concat_last_four')
# doc_sim_score_with_cl(p, r, comp_type='wmd')

100%|██████████| 2484/2484 [00:45<00:00, 54.26it/s]


In [23]:
scores.sort(key=lambda x: x[2], reverse=True)

In [24]:
# [s for s in scores if s[1] == 8]
scores

[(0, 1342, 11.28521728515625),
 (0, 1510, 10.982083320617676),
 (0, 636, 10.79155445098877),
 (0, 1519, 10.541608333587646),
 (0, 1106, 10.193195343017578),
 (0, 2261, 10.160942554473877),
 (0, 661, 10.126782417297363),
 (0, 585, 10.067670345306396),
 (0, 1333, 10.060352802276611),
 (0, 853, 10.007856369018555),
 (0, 1699, 9.96583080291748),
 (0, 798, 9.957141876220703),
 (0, 2016, 9.948312282562256),
 (0, 805, 9.945833206176758),
 (0, 1050, 9.937272548675537),
 (0, 595, 9.901795864105225),
 (0, 540, 9.871129989624023),
 (0, 453, 9.863186836242676),
 (0, 454, 9.76578664779663),
 (0, 990, 9.756860256195068),
 (0, 1490, 9.718706607818604),
 (0, 1509, 9.718706607818604),
 (0, 1662, 9.712028503417969),
 (0, 804, 9.69934368133545),
 (0, 1331, 9.690459728240967),
 (0, 1206, 9.670851230621338),
 (0, 427, 9.670438289642334),
 (0, 1336, 9.636992454528809),
 (0, 898, 9.627134799957275),
 (0, 452, 9.622880458831787),
 (0, 2183, 9.604079723358154),
 (0, 1076, 9.599502086639404),
 (0, 239, 9.580161

In [25]:
postings[3]

'Senior Associate Attorney - Elder Law / Trusts and Estates  Our legal team is committed to providing each client with quality counsel, innovative solutions, and personalized service. Founded in 2000, the firm offers the legal expertise of its 115+ attorneys, who have accumulated experience and problem-solving skills over decades of practice.\nWe are a prominent Lake Success Law Firm seeking an associate attorney for its growing Elder Law and Estate Planning practice. The successful candidate will be a self-motivated, detail-oriented team member with strong communication skills and a desire to grow their practice. Experience with Estate Planning, Administration, and Litigation and is preferred.\n Responsibilities will include:\nCounseling clients with regard to estate planning and asset protection;Formulating and overseeing execution of Medicaid and estate plans;Drafting wills, revocable and irrevocable trusts, powers of attorney, health care proxies, and living wills;Estate Administra

In [ ]:
resumes[743]# doc_sim_score(p, r, comp_type='pairwise_cosine')

###### torch.set_float32_matmul_precision('highest')
ll = nn.TripletMarginLoss(margin=2, reduction='mean')
x = torch.randn(16, 20, 128)
y = torch.randn(16, 59, 128)
z = torch.randn(16, 78, 128)
# ll(x,y,z)

In [152]:
# torch.matmul(x, y.transpose(1,2))
mask = torch.ones(16, 20, 128, dtype=bool)
mask[:]
X = torch.randn(16, 20, 128)
X

tensor([[[-0.5074,  0.6456,  0.0627,  ...,  1.4339,  1.4598,  0.6911],
         [ 0.6769, -0.6305,  1.4965,  ...,  0.3071, -0.5117, -0.2067],
         [-1.4282, -0.4100,  0.4808,  ..., -0.5921, -1.0037,  1.1876],
         ...,
         [-0.0815,  0.9946,  0.6107,  ...,  0.4782, -1.3927,  0.3651],
         [-0.9729,  1.3114,  0.6697,  ...,  1.2307,  0.7235, -0.6874],
         [-0.5364, -1.0250, -0.2753,  ..., -0.2468, -0.3026,  2.1389]],

        [[ 0.2693, -1.6578,  0.2199,  ...,  0.5011,  1.1775,  0.9088],
         [ 1.6300,  0.5477, -0.6694,  ..., -0.2469, -0.0966, -2.0516],
         [ 0.2596,  1.3851, -0.8337,  ...,  0.3786,  0.0634,  0.1645],
         ...,
         [ 1.0545, -2.9513, -1.6314,  ...,  0.9906,  0.9881,  1.0028],
         [-0.1108,  0.7447, -1.4254,  ...,  0.2376, -1.1245,  0.3419],
         [ 0.1021,  0.9599,  1.0310,  ...,  2.1958,  0.7041, -1.6799]],

        [[-1.6297,  0.1698, -0.5317,  ..., -1.2757, -0.7109,  1.3798],
         [ 0.6343, -0.9593, -1.4659,  ..., -0

In [153]:
lm_dataset = load_from_disk('./temp/lm_dataset')
dataset = ContrastiveLearningDataset(lm_dataset, 'train', TOKENIZER, AUGMENTATION_FNS, num_sample=5000)


PicklingError: Can't pickle <function sentence_dropout at 0x15523127f130>: it's not the same object as contrastive_learning.sentence_dropout

In [ ]:
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
split_triplet_data = lambda x: (BatchEncoding({k: v[:, i] for k,v in x.items()}) for i in range(3))
anchor, pos, neg = split_triplet_data(next(iter(loader)))

In [ ]:
anchor['attention_mask']#.unsqueeze(2)#.expand(-1, -1, 384).shape

In [ ]:
# data = TOKENIZER(postings[48], truncation=True, padding='max_length', max_length=510+2, return_tensors='pt').to(DEVICE)
Z = cl_model(anchor.to(DEVICE))
Z

In [ ]:
res = torch.where(anchor['attention_mask'].to(bool).unsqueeze(2), Z, torch.nan).nanmean(dim=1)

In [ ]:
res1 = torch.empty(16, 384)
for batch in range(Z.shape[0]):
    for embed_ind in range(Z.shape[2]):
        res1[batch, embed_ind] = torch.where(anchor['attention_mask'][batch].to(bool), Z[batch, :, embed_ind], torch.nan).nanmean()
#         if anchor['attention_mask'][batch].sum


In [ ]:
torch.allclose(res.cpu(),res1)

In [ ]:
dist = lambda a,b: torch.norm(a-b, dim=2)
(dist(x,y) - dist(x,z) + 2).mean()


In [ ]:
doc_sim_score_with_cl(cl_model, postings[29], resumes[4], 'cuda:0')

In [ ]:
from datasets import load_dataset


In [ ]:
dataset = load_dataset('csv', data_files={'train': './linkedin_data/postings.csv'})

In [ ]:
dataset['train'][0]

In [ ]:

# import pandas as pd
# df = pd.read_csv('linkedin_data/postings.csv')
# df.sample(n=5000, random_state=1).to_csv('temp/postings5k.csv')